# Genie verification

## 1. simple questions for chat

### whats the gender distribution?

In [0]:
%sql
USE CATALOG marathos;
USE SCHEMA gold;

SELECT
  athlete_gender,
  COUNT(*) AS amount
FROM
  mart_general
GROUP BY
  athlete_gender
ORDER BY
  amount DESC;

In [0]:
%sql
SELECT
  `athlete_gender`,
  COUNT(*) AS gender_count
FROM
  `marathos`.`gold`.`mart_general`
WHERE
  `athlete_gender` IS NOT NULL
GROUP BY
  `athlete_gender`
ORDER BY
  gender_count DESC;

### Whats the top 10 countries in mart_time_event?

In [0]:
%sql
SELECT 
    country_name,
    COUNT(*) AS number_of_finishers
FROM marathos.gold.mart_time_event
GROUP BY country_name
ORDER BY number_of_finishers DESC
LIMIT 10


In [0]:
%sql
WITH ranked_countries AS (
  SELECT
    `country_name`,
    COUNT(*) AS event_count,
    RANK() OVER (ORDER BY COUNT(*) DESC) AS rank
  FROM
    `marathos`.`gold`.`mart_time_event`
  WHERE
    `country_name` IS NOT NULL
  GROUP BY
    `country_name`
)
SELECT
  `country_name`,
  event_count
FROM
  ranked_countries
WHERE
  rank <= 10
ORDER BY
  event_count DESC;

### Which country has the highest average speed in mart_distance_event?

In [0]:
%sql
SELECT 
    country_name,
    ROUND(AVG(average_speed), 2) AS avg_speed
FROM marathos.gold.mart_distance_event
WHERE country_name IS NOT NULL
AND average_speed IS NOT NULL
GROUP BY country_name
ORDER BY avg_speed DESC
LIMIT 10

In [0]:
%sql
WITH avg_speed_by_country AS (
  SELECT
    `country_name`,
    AVG(`average_speed`) AS avg_speed,
    RANK() OVER (ORDER BY AVG(`average_speed`) DESC) AS rank
  FROM
    `marathos`.`gold`.`mart_distance_event`
  WHERE
    `country_name` IS NOT NULL
    AND `average_speed` IS NOT NULL
  GROUP BY
    `country_name`
)
SELECT
  `country_name`,
  avg_speed
FROM
  avg_speed_by_country
WHERE
  rank = 1;

### How many athletes finished a 100km race?

In [0]:
%sql
SELECT COUNT(*) AS number_of_finishers
FROM marathos.gold.mart_distance_event
WHERE event_distance_km = 100

Genies answer: A total of 852,111 athletes finished a 100km race, based on the data from the mart_distance_event table. This figure represents all recorded completions for races labeled as 100km.

Would you like me to count athletes who finished races exactly 100 kilometers long or include those with distances approximately 100 kilometers as well?

My answer: exactly 100 km

Genies answer: A total of 852,088 athletes finished races that were exactly 100 kilometers long. This count is based strictly on events with a distance of 100.0 km.

## 2. Advanced questions for agent

### Which country has improved their average speed the most over the last 10 years?

In [0]:
%sql
WITH speed_by_year AS (
    SELECT 
        country_name,
        year_of_event,
        ROUND(AVG(average_speed), 3) AS avg_speed
    FROM marathos.gold.mart_distance_event
    WHERE country_name IS NOT NULL
    AND average_speed IS NOT NULL
    AND year_of_event >= 2013
    GROUP BY country_name, year_of_event
),
first_and_last AS (
    SELECT
        country_name,
        MIN_BY(avg_speed, year_of_event) AS speed_2013,
        MAX_BY(avg_speed, year_of_event) AS speed_latest
    FROM speed_by_year
    GROUP BY country_name
)
SELECT
    country_name,
    ROUND(speed_latest - speed_2013, 3) AS speed_improvement
FROM first_and_last
WHERE speed_2013 IS NOT NULL
AND speed_latest IS NOT NULL
ORDER BY speed_improvement DESC
LIMIT 10

In [0]:
%sql
WITH country_speeds AS (
  SELECT
    `country_name`,
    `year_of_event`,
    AVG(`average_speed`) as avg_speed
  FROM
    `marathos`.`gold`.`mart_general`
  WHERE
    `year_of_event` BETWEEN 2013 AND 2022
    AND `country_name` IS NOT NULL
    AND `average_speed` IS NOT NULL
  GROUP BY
    `country_name`,
    `year_of_event`
),
first_last_years AS (
  SELECT
    `country_name`,
    MAX(
      CASE
        WHEN `year_of_event` = 2013 THEN avg_speed
      END
    ) as speed_2013,
    MAX(
      CASE
        WHEN `year_of_event` = 2022 THEN avg_speed
      END
    ) as speed_2022
  FROM
    country_speeds
  GROUP BY
    `country_name`
  HAVING
    speed_2013 IS NOT NULL
    AND speed_2022 IS NOT NULL
)
SELECT
  `country_name`,
  speed_2013,
  speed_2022,
  (speed_2022 - speed_2013) as speed_improvement,
  ROUND(((speed_2022 - speed_2013) / speed_2013) * 100, 2) as percent_improvement
FROM
  first_last_years
ORDER BY
  speed_improvement DESC
LIMIT 20

In [0]:
%sql
WITH country_speeds AS (
  SELECT
    `country_name`,
    `year_of_event`,
    AVG(`average_speed`) as avg_speed
  FROM
    `marathos`.`gold`.`mart_general`
  WHERE
    `year_of_event` BETWEEN 2013 AND 2022
    AND `country_name` IN ('Rwanda', 'Cape Verde', 'Burkina Faso', 'Lesotho', 'Liechtenstein')
    AND `average_speed` IS NOT NULL
  GROUP BY
    `country_name`,
    `year_of_event`
)
SELECT
  `country_name`,
  `year_of_event`,
  avg_speed
FROM
  country_speeds
ORDER BY
  `country_name`,
  `year_of_event`